<a href="https://colab.research.google.com/github/Suim00/hdac-knhanes-skipbreakfast-obesity/blob/main/code/09_data_check_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9주차 자료 확인 (9팀)


## 1. 내 Google Drive 연결 및 자료 읽기

In [15]:
# 1) 내 Google Drive 연결 - 원시자료는 각자 자기 Drive에 둡니다
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data/HN24_ALL(SAS)'   # 팀 공통 경로
DATA_FILE = f'{DATA_DIR}/hn24_all.sas7bdat'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# 2) 필요한 변수만 읽기
!pip -q install pyreadstat
import pyreadstat
import pandas as pd

# 변수 정의표의 원변수명 (이용지침서 표기)
need = ['age', 'sex', 'HE_prg', 'L_BR_FQ', 'HE_ht', 'HE_wt', 'HE_BMI', 'HE_obe', 'wt_tot', 'kstrata', 'psu']

# 파일의 전체 변수 목록만 먼저 읽기 (자료 행은 읽지 않음)
_, meta0 = pyreadstat.read_sas7bdat(DATA_FILE, metadataonly=True)
all_cols = list(meta0.column_names)
print('파일 전체 변수 수:', len(all_cols))

파일 전체 변수 수: 813


## 2. 변수 존재 여부

In [17]:
lower_map = {c.lower(): c for c in all_cols}
present = {v: lower_map.get(v.lower()) for v in need}
for v, actual in present.items():
    print(f'{v:10s} ->', actual if actual else '파일에 없음')

missing_vars = [v for v, a in present.items() if a is None]
print()
print('없는 변수:', missing_vars if missing_vars else '없음')

age        -> age
sex        -> sex
HE_prg     -> HE_prg
L_BR_FQ    -> L_BR_FQ
HE_ht      -> HE_ht
HE_wt      -> HE_wt
HE_BMI     -> HE_BMI
HE_obe     -> HE_obe
wt_tot     -> wt_tot
kstrata    -> kstrata
psu        -> psu

없는 변수: 없음


In [18]:
# 파일에 있는 이름으로 변수만 골라 읽기
use_cols = [a for a in present.values() if a]
df, meta = pyreadstat.read_sas7bdat(DATA_FILE, usecols=use_cols)
# 이후 코드에서 쓰기 쉽도록 정의표의 이름으로 통일
df = df.rename(columns={a: v for v, a in present.items() if a})
print(df.shape)   # 행 수와 열 수만 출력

(6997, 11)


## 3. 원 코드별 빈도표

In [19]:
# 노출: 아침식사 빈도 (L_BR_FQ) - 1 주5~7회, 2 주3~4회, 3 주1~2회, 4 거의 안 한다, 9 모름/무응답
print('L_BR_FQ 원 코드별 인원 (결측 포함)')
print(df['L_BR_FQ'].value_counts(dropna=False).sort_index())

L_BR_FQ 원 코드별 인원 (결측 포함)
L_BR_FQ
1.0    4214
2.0     669
3.0     786
4.0    1136
NaN     192
Name: count, dtype: int64


In [20]:
# 결과: 비만 유병여부 (HE_obe) - 1 저체중, 2 정상, 3 비만전단계, 4 1단계, 5 2단계, 6 3단계 (비임신자만 값 있음)
print('HE_obe 원 코드별 인원 (결측 포함)')
print(df['HE_obe'].value_counts(dropna=False).sort_index())

HE_obe 원 코드별 인원 (결측 포함)
HE_obe
1.0     285
2.0    2134
3.0    1329
4.0    1751
5.0     344
6.0      66
NaN    1088
Name: count, dtype: int64


In [21]:
# 성별 (sex) - 1 남자, 2 여자
print('sex 원 코드별 인원')
print(df['sex'].value_counts(dropna=False).sort_index())

sex 원 코드별 인원
sex
1.0    3070
2.0    3927
Name: count, dtype: int64


In [22]:
# 임신여부 (HE_prg) - 0 아니오, 1 예, 8 비해당(남자)
print('HE_prg 원 코드별 인원 (결측 포함)')
print(df['HE_prg'].value_counts(dropna=False).sort_index())

HE_prg 원 코드별 인원 (결측 포함)
HE_prg
0.0    3911
1.0      16
8.0    3070
Name: count, dtype: int64


## 4. 가중치·층·집락 정보

In [23]:
print('wt_tot 결측 인원:', int(df['wt_tot'].isna().sum()))
print('wt_tot 0 이하 인원:', int((df['wt_tot'] <= 0).sum()))
print()
print('wt_tot 요약 (집계값)')
print(df['wt_tot'].describe())
print()
print('kstrata 층 수:', df['kstrata'].nunique(dropna=True), '/ 결측:', int(df['kstrata'].isna().sum()))
print('psu 집락 수:', df['psu'].nunique(dropna=True), '/ 결측:', int(df['psu'].isna().sum()))

wt_tot 결측 인원: 192
wt_tot 0 이하 인원: 0

wt_tot 요약 (집계값)
count     6805.000000
mean      7571.202792
std       3537.288651
min        388.119306
25%       5138.748407
50%       6786.485519
75%       9405.572181
max      19285.643829
Name: wt_tot, dtype: float64

kstrata 층 수: 27 / 결측: 0
psu 집락 수: 192 / 결측: 0


## 5. 분석 대상 선정 단계별 인원

- 0단계: 전체 조사 참여자
- 1단계: 만 19세 이상 성인
- 2단계: 임신 중인 여성 제외 (`HE_prg` = 1)
- 3단계: 아침식사 빈도 응답자 (모름·무응답 제외)
- 4단계: 신장·체중 응답자 (이상값 제외)
- 5단계: 가중치·층·집락 변수가 유효한 자료

In [24]:
# 4단계 이상값 확인용 - 신장·체중·BMI 분포 요약 (분석 대상 중 3단계까지 통과한 사람, 집계값)
chk = (df['age'] >= 19) & (df['HE_prg'] != 1) & df['L_BR_FQ'].isin([1, 2, 3, 4])
print(df.loc[chk, ['HE_ht', 'HE_wt', 'HE_BMI']].describe().round(1))

        HE_ht   HE_wt  HE_BMI
count  5749.0  5820.0  5749.0
mean    163.4    64.8    24.1
std       9.4    13.6     3.9
min     127.9    27.5    13.1
25%     156.7    54.9    21.5
50%     163.0    63.0    23.7
75%     170.0    72.7    26.3
max     194.0   147.0    48.7


In [25]:
# 4단계의 이상값 기준: 이용지침서에는 신장·체중 이상값 제외 기준이 없다.
# 바로 위 분포를 보고 팀에서 기준을 정하면 아래 값을 채운다. None이면 이상값 기준으로는 제외하지 않는다.
BMI_MIN = None   # 예: 팀이 정한 BMI 하한
BMI_MAX = None   # 예: 팀이 정한 BMI 상한

In [26]:
steps = []
cur = pd.Series(True, index=df.index)

def add_step(name, cond):
    global cur
    before = int(cur.sum())
    cur = cur & cond
    after = int(cur.sum())
    steps.append({'단계': name, '제외된 인원': before - after, '남은 인원': after})

steps.append({'단계': '0. 전체 조사 참여자', '제외된 인원': 0, '남은 인원': int(cur.sum())})
add_step('1. 만 19세 이상 성인', df['age'] >= 19)
add_step('2. 임신 중인 여성 제외 (HE_prg = 1)', df['HE_prg'] != 1)
add_step('3. 아침식사 빈도 응답자 (L_BR_FQ 1~4)', df['L_BR_FQ'].isin([1, 2, 3, 4]))

ok_body = df['HE_ht'].notna() & df['HE_wt'].notna() & df['HE_BMI'].notna()
if BMI_MIN is not None:
    ok_body = ok_body & (df['HE_BMI'] >= BMI_MIN)
if BMI_MAX is not None:
    ok_body = ok_body & (df['HE_BMI'] <= BMI_MAX)
add_step('4. 신장·체중 응답자 (이상값 제외)', ok_body)

ok_design = (df['wt_tot'].notna() & (df['wt_tot'] > 0)
             & df['kstrata'].notna() & df['psu'].notna())
add_step('5. 가중치·층·집락 변수 유효', ok_design)

flow = pd.DataFrame(steps)
print(flow.to_string(index=False))
print()
print('최종 분석대상:', int(cur.sum()))
print('최종 분석대상 중 HE_obe 결측:', int(df.loc[cur, 'HE_obe'].isna().sum()))

                          단계  제외된 인원  남은 인원
                0. 전체 조사 참여자       0   6997
              1. 만 19세 이상 성인     964   6033
 2. 임신 중인 여성 제외 (HE_prg = 1)      16   6017
3. 아침식사 빈도 응답자 (L_BR_FQ 1~4)     174   5843
       4. 신장·체중 응답자 (이상값 제외)      94   5749
           5. 가중치·층·집락 변수 유효       0   5749

최종 분석대상: 5749
최종 분석대상 중 HE_obe 결측: 0


## 5-1. 분석 대상 안에서의 아침식사 빈도
5번 셀에서 만든 최종 분석대상(`cur`) 안에서 `L_BR_FQ` 코드별 인원을 확인한다. 노출군과 비교군의 범주를 정하는 근거로 쓴다.
(표본 인원 기준이며 가중치를 적용하지 않은 값이다.)

In [27]:
sub = df[cur].copy()
print('분석 대상 인원:', len(sub))
print()

cnt = sub['L_BR_FQ'].value_counts().sort_index()
pct = (cnt / cnt.sum() * 100).round(1)
print('L_BR_FQ 코드별 인원 (분석 대상)')
print(pd.DataFrame({'인원': cnt.astype(int), '비율(%)': pct}))

분석 대상 인원: 5749

L_BR_FQ 코드별 인원 (분석 대상)
           인원  비율(%)
L_BR_FQ             
1.0      3499   60.9
2.0       564    9.8
3.0       665   11.6
4.0      1021   17.8


In [28]:
# 코드별 비만(HE_obe 4~6, BMI 25 이상) 인원 - 가중치를 적용하지 않은 단순 집계 (설명용)
sub['obese'] = (sub['HE_obe'] >= 4).astype(int)
tab = pd.crosstab(sub['L_BR_FQ'], sub['obese'])
tab.columns = ['비비만(BMI<25)', '비만(BMI>=25)']
tab['합계'] = tab.sum(axis=1)
tab['비만 비율(%)'] = (tab['비만(BMI>=25)'] / tab['합계'] * 100).round(1)
print(tab)

         비비만(BMI<25)  비만(BMI>=25)    합계  비만 비율(%)
L_BR_FQ                                          
1.0             2271         1228  3499      35.1
2.0              329          235   564      41.7
3.0              399          266   665      40.0
4.0              651          370  1021      36.2
